In [1]:
import cvxpy as cp
import numpy as np

# 假设 num_links 和 T_max 已经定义
num_links = 10
T_max = 50
Delta_t = 1
B_total = 100
P_max = 10
N0 = 1e-9
I = np.random.rand(num_links, T_max)  # 假设干扰是已知的
g = np.random.rand(num_links, T_max)  # 信道增益

# 决策变量
p = cp.Variable((num_links, T_max), nonneg=True)  # 功率分配
b = cp.Variable((num_links, T_max), nonneg=True)  # 带宽分配
r = cp.Variable((num_links, T_max), nonneg=True)  # 传输速率

# 目标函数：最小化通信能耗
objective = cp.Minimize(cp.sum(cp.multiply(p, Delta_t)))

# 约束条件
constraints = []

# 1. 确保总带宽分配不超过节点容量
for t in range(T_max):
    constraints.append(cp.sum(b[:, t]) <= B_total)  # 带宽限制

# 2. 确保所有任务数据在给定工期内完成传输
O_v = np.random.rand(num_links)  # 假设每个任务的数据量是已知的
for i in range(num_links):
    constraints.append(cp.sum(r[i, :] * Delta_t) >= O_v[i])  # 数据传输完成

# 3. 分段线性化约束
K = 4  # 分段数量
phi_bounds = np.linspace(0, 10, K+1)  # 分段边界
alpha = np.random.rand(K)  # 线性近似的斜率
beta = np.random.rand(K)  # 线性近似的截距
delta = cp.Variable((num_links, T_max, K), boolean=True)  # 二进制变量

M = 1e6  # 大M常数
for i in range(num_links):
    for t in range(T_max):
        phi = cp.multiply(p[i, t], g[i, t]) / (N0 + I[i, t])
        
        # 分段的线性化约束
        constraints.append(cp.sum(delta[i, t, :]) == 1)  # 只有一个段是活跃的
        for k in range(K):
            constraints.append(phi_bounds[k] * delta[i, t, k] <= phi)
            constraints.append(phi <= phi_bounds[k+1] * delta[i, t, k])
            constraints.append(r[i, t] <= b[i, t] * (alpha[k] * phi + beta[k]) + M * (1 - delta[i, t, k]))

# 4. 功率和带宽非负约束
constraints.append(p <= P_max)

# 求解问题
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.GUROBI)

# 输出结果
if problem.status == cp.OPTIMAL:
    print("Optimal Solution Found!")
    p_star = p.value
    b_star = b.value
    r_star = r.value
    print("Optimal power allocation:", p_star)
    print("Optimal bandwidth allocation:", b_star)
    print("Optimal rate:", r_star)
else:
    print("Problem is infeasible or unbounded.")


ValueError: Expressions of dimension greater than 2 are not supported.

Bad pipe message: %s [b'\x08\x1f)\x9d\x03\x17\x1b\xe6\x8a\\\xab?~\xb1\x96\x8f\xfc~\x00\x02\xbc\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00\x1a\x00\x1b\x00\x1e\x00\x1f\x00 \x00!\x00"\x00#\x00$\x00%\x00&\x00\'\x00(\x00)\x00*\x00+\x00,\x00-\x00.\x00/\x000\x001\x002\x003\x004\x005\x006\x007\x008\x009\x00:\x00;\x00<\x00=\x00>\x00?\x00@\x00A\x00B\x00C\x00D\x00E\x00F\x00g\x00h\x00i\x00j\x00k\x00l\x00m\x00\x84\x00\x85\x00\x86\x00\x87\x00\x88\x00\x89\x00\x8a\x00\x8b\x00\x8c\x00\x8d\x00\x8e\x00\x8f\x00\x90\x00\x91\x00\x92\x00\x93\x00\x94\x00\x95\x00\x96\x00\x97\x00\x98\x00\x99\x00\x9a\x00\x9b\x00\x9c\x00\x9d\x00\x9e\x00\x9f\x00\xa0\x00\xa1\x00\xa2\x00\xa3']
Bad pipe message: %s [b'\xea\x8a\xfbT\x8c\xd4\xa5R\x8bR-Q\x99!\x0cU\x1d\xc8\x00\x02\xbc\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x0